In [1]:
import pandas as pd

In [2]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [3]:
model = pd.read_csv("/data/aman_singh/acuuracy_check/all_channels brand_asm mar'26.csv")
missing = pd.read_csv("/data/aman_singh/acuuracy_check/missing_df_all_brand_asm.csv")

In [4]:
print(model['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7' 'M+8' 'M+9' 'M+10'
 'M+11' 'M+12']


In [5]:
print(missing['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7' 'M+8' 'M+9' 'M+10'
 'M+11' 'M+12']


In [ ]:
#missing = missing[missing['M month'].notna()]

In [6]:
print(missing['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7' 'M+8' 'M+9' 'M+10'
 'M+11' 'M+12']


In [7]:
model['skipped'] = 0
missing['skipped'] = 1

In [8]:
all = pd.concat([
    model, missing
], ignore_index=True)

In [9]:
all['channel'].unique()

array(['ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [10]:
all.duplicated(subset=['channel', 'key', 'month_date']).sum()

0

In [ ]:
# all.to_csv("ALL COMBINATIONS gt Live Run jan'25.csv", index=False)

In [ ]:
# all.to_csv('All_combination_Nov25_live_with_festivals.csv', index=False)

In [11]:
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,class,skipped,extra_vol_co,other_co,price_off_co,seasonal_month_flag
0,BCE1_ADV-AHO-R,2023-01-31,15.0,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,NaN,NaN,NaN,NaN,B,0,NaN,NaN,NaN,NaN
1,BCE1_ADV-AHO-R,2023-02-28,15.0,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,NaN,0.001512,NaN,NaN,B,0,NaN,NaN,NaN,NaN
2,BCE1_ADV-AHO-R,2023-03-31,15.0,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,NaN,0.001364,0.001512,NaN,B,0,NaN,NaN,NaN,NaN
3,BCE1_ADV-AHO-R,2023-04-30,15.0,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,NaN,0.000810,0.001364,0.001512,B,0,NaN,NaN,NaN,NaN
4,BCE1_ADV-AHO-R,2023-05-31,9.0,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,NaN,0.000230,0.000810,0.001364,B,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191094,QCE2_SW_SGPRF,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0
191095,QCE2_SW_SGPRF,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0
191096,QCE2_SW_SGPRF,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0
191097,QCE2_SW_SGPRF,2027-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0


In [12]:
all.columns[:60]

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'brand_code', 'diwali',
       'diwali_lead_1', 'diwali_lead_2', 'ganesh_chaturthi',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_2',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'drive', 'outlier', 'run_month', 'M month',
       'pred_prophet_60%ile', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_valu

In [13]:
all.columns[60:]

Index(['Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'class', 'skipped', 'extra_vol_co', 'other_co',
       'price_off_co', 'seasonal_month_flag'],
      dtype='object')

In [14]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = all.groupby(
    ['channel','key',"run_month"]
).apply(detect_trend_for_group).reset_index()

In [15]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,channel,key,run_month,trend_flag,p3m_slope_flag,final_trend
0,ECOM,BCE1_ADV-AHO-R,2026-03-31,0,1,0
1,ECOM,BCE1_BIO OILS,2026-03-31,0,1,0
2,ECOM,BCE1_BRD_DOGAS,2026-03-31,-1,0,0
3,ECOM,BCE1_BRD_FSWSH,2026-03-31,-1,0,0
4,ECOM,BCE1_BRD_HROIL,2026-03-31,-1,0,0
...,...,...,...,...,...,...
4455,QCOM,QCW2_SW HRGEL,2026-03-31,1,-1,0
4456,QCOM,QCW2_SW HSPRY,2026-03-31,1,-1,0
4457,QCOM,QCW2_SW STLDEO,2026-03-31,1,-1,0
4458,QCOM,QCW2_SW_HR_WAX,2026-03-31,1,1,1


In [16]:
trend_df[trend_df['final_trend'] == 1]

,channel,key,run_month,trend_flag,p3m_slope_flag,final_trend
20,ECOM,BCE1_PADV-HRCR,2026-03-31,1,1,1
28,ECOM,BCE1_P_AL_GOLD,2026-03-31,1,1,1
31,ECOM,BCE1_P_EN_RSMR,2026-03-31,1,1,1
43,ECOM,BCE2_ADV-AHO-R,2026-03-31,1,1,1
50,ECOM,BCE2_MALO-NATU,2026-03-31,1,1,1
...,...,...,...,...,...,...
4441,QCOM,QCW2_SAFF OATS,2026-03-31,1,1,1
4442,QCOM,QCW2_SAFF SALT,2026-03-31,1,1,1
4445,QCOM,QCW2_SAF_HONEY,2026-03-31,1,1,1
4449,QCOM,QCW2_SFOATS-FL,2026-03-31,1,1,1


## detect seasonality

In [18]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = all.groupby(
    ['channel','asm_area_code','brand_code',"run_month"]
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: in

,channel,asm_area_code,brand_code,run_month,seasonality_flag
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,0
1,ECOM,BCE1,BIO OILS,2026-03-31,0
2,ECOM,BCE1,BRD_DOGAS,2026-03-31,0
3,ECOM,BCE1,BRD_FSWSH,2026-03-31,0
4,ECOM,BCE1,BRD_HROIL,2026-03-31,0
...,...,...,...,...,...
4455,QCOM,QCW2,SW HRGEL,2026-03-31,0
4456,QCOM,QCW2,SW HSPRY,2026-03-31,0
4457,QCOM,QCW2,SW STLDEO,2026-03-31,0
4458,QCOM,QCW2,SW_HR_WAX,2026-03-31,0


In [19]:
seasonality_df[seasonality_df['seasonality_flag'] == 1]

,channel,asm_area_code,brand_code,run_month,seasonality_flag
19,ECOM,BCE1,PADV-HOT,2026-03-31,1
58,ECOM,BCE2,PADV-HOT,2026-03-31,1
60,ECOM,BCE2,PADVJAS-R,2026-03-31,1
98,ECOM,BCN1,PADV-HOT,2026-03-31,1
103,ECOM,BCN1,PA_CN_HO,2026-03-31,1
...,...,...,...,...,...
4340,QCOM,QCW1,PADV-HOT,2026-03-31,1
4362,QCOM,QCW1,SAF-MUSLI,2026-03-31,1
4368,QCOM,QCW1,SAFF_ODLS,2026-03-31,1
4410,QCOM,QCW2,PADV-HOT,2026-03-31,1


In [18]:
seasonality_df.to_csv('seasonality_all2.csv')

In [25]:
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,grp_pt,pt_eve,npt_eve,print_spends_in_lacs,tv_spends_in_lacs,radio_spends_in_lacs,npt,pt,npd,update_timestamp
0,AURG_D3A4_808712,2023-01-31,0.028333,0.016833,0.050077,0.02664,0.000610,0.000362,0.001078,0.000574,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AURG_D3A4_808712,2023-02-28,0.028333,0.016833,0.000000,0.02648,0.000610,0.000362,0.000000,0.000570,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AURG_D3A4_808712,2023-03-31,0.028333,0.016833,0.031957,0.02228,0.000610,0.000362,0.000688,0.000480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AURG_D3A4_808712,2023-04-30,0.028333,0.016833,0.000000,0.01102,0.000610,0.000362,0.000000,0.000237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AURG_D3A4_808712,2023-05-31,0.020000,0.016833,0.009372,0.00930,0.000431,0.000362,0.000202,0.000200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16232,WUP_D113_808731,2026-02-28,0.000000,0.001000,0.000000,0.00080,0.000000,0.000022,0.000000,0.000017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16233,WUP_D113_808731,2026-03-31,0.000000,0.001000,0.000000,0.00000,0.000000,0.000022,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16234,WUP_D113_808731,2026-04-30,0.000000,0.001000,0.000000,0.00000,0.000000,0.000022,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16235,WUP_D113_808731,2026-05-31,0.000000,0.001000,0.000000,0.00000,0.000000,0.000022,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = all.groupby(
    ["channel", "key","run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,channel,key,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,ECOM,BCE1_ADV-AHO-R,2026-03-31,0.0,0.002300,0.000736,0.000521
1,ECOM,BCE1_BIO OILS,2026-03-31,0.0,0.007532,0.001300,0.002077
2,ECOM,BCE1_BRD_DOGAS,2026-03-31,0.0,0.000546,0.000050,0.000166
3,ECOM,BCE1_BRD_FSWSH,2026-03-31,0.0,0.000388,0.000035,0.000117
4,ECOM,BCE1_BRD_HROIL,2026-03-31,0.0,0.000561,0.000051,0.000170


In [21]:
trend_df = trend_df.merge(threshold_df, on = ["channel", "key","run_month"], how = 'left')
trend_df

,channel,key,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,ECOM,BCE1_ADV-AHO-R,2026-03-31,0,1,0,0.000000,0.002300,0.000736,0.000521
1,ECOM,BCE1_BIO OILS,2026-03-31,0,1,0,0.000000,0.007532,0.001300,0.002077
2,ECOM,BCE1_BRD_DOGAS,2026-03-31,-1,0,0,0.000000,0.000546,0.000050,0.000166
3,ECOM,BCE1_BRD_FSWSH,2026-03-31,-1,0,0,0.000000,0.000388,0.000035,0.000117
4,ECOM,BCE1_BRD_HROIL,2026-03-31,-1,0,0,0.000000,0.000561,0.000051,0.000170
...,...,...,...,...,...,...,...,...,...,...
4455,QCOM,QCW2_SW HRGEL,2026-03-31,1,-1,0,0.013048,0.056606,0.030471,0.008712
4456,QCOM,QCW2_SW HSPRY,2026-03-31,1,-1,0,0.007287,0.022805,0.013494,0.003104
4457,QCOM,QCW2_SW STLDEO,2026-03-31,1,-1,0,0.004047,0.023284,0.011741,0.003847
4458,QCOM,QCW2_SW_HR_WAX,2026-03-31,1,1,1,0.002540,0.009439,0.005300,0.001380


In [17]:
trend_df.to_csv('t_thres_df_2.csv')

In [22]:
all.shape

(191099, 69)

In [23]:
all = all.merge(seasonality_df, on = ['channel','asm_area_code', 'brand_code',"run_month"], how = 'left')
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,class,skipped,extra_vol_co,other_co,price_off_co,seasonal_month_flag,seasonality_flag
0,BCE1_ADV-AHO-R,2023-01-31,15.0,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,NaN,NaN,NaN,B,0,NaN,NaN,NaN,NaN,0
1,BCE1_ADV-AHO-R,2023-02-28,15.0,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,0.001512,NaN,NaN,B,0,NaN,NaN,NaN,NaN,0
2,BCE1_ADV-AHO-R,2023-03-31,15.0,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,0.001364,0.001512,NaN,B,0,NaN,NaN,NaN,NaN,0
3,BCE1_ADV-AHO-R,2023-04-30,15.0,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,0.000810,0.001364,0.001512,B,0,NaN,NaN,NaN,NaN,0
4,BCE1_ADV-AHO-R,2023-05-31,9.0,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,0.000230,0.000810,0.001364,B,0,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191094,QCE2_SW_SGPRF,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0,0
191095,QCE2_SW_SGPRF,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0,0
191096,QCE2_SW_SGPRF,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0,0
191097,QCE2_SW_SGPRF,2027-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,NPD,1,0.0,0.0,0.0,0.0,0


In [24]:
trend_df.columns

Index(['channel', 'key', 'run_month', 'trend_flag', 'p3m_slope_flag',
       'final_trend', 'lower_threshold', 'upper_threshold', 'mean_value',
       'std_value'],
      dtype='object')

In [25]:
all = all.merge(trend_df[["channel", "key","run_month",
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["channel", "key","run_month"], how = 'left')
all


,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,class,skipped,extra_vol_co,other_co,price_off_co,seasonal_month_flag,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,BCE1_ADV-AHO-R,2023-01-31,15.0,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,B,0,NaN,NaN,NaN,NaN,0,0,0.0,0.002300
1,BCE1_ADV-AHO-R,2023-02-28,15.0,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,B,0,NaN,NaN,NaN,NaN,0,0,0.0,0.002300
2,BCE1_ADV-AHO-R,2023-03-31,15.0,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,B,0,NaN,NaN,NaN,NaN,0,0,0.0,0.002300
3,BCE1_ADV-AHO-R,2023-04-30,15.0,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,B,0,NaN,NaN,NaN,NaN,0,0,0.0,0.002300
4,BCE1_ADV-AHO-R,2023-05-31,9.0,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,B,0,NaN,NaN,NaN,NaN,0,0,0.0,0.002300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191094,QCE2_SW_SGPRF,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NPD,1,0.0,0.0,0.0,0.0,0,0,0.0,0.001503
191095,QCE2_SW_SGPRF,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NPD,1,0.0,0.0,0.0,0.0,0,0,0.0,0.001503
191096,QCE2_SW_SGPRF,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NPD,1,0.0,0.0,0.0,0.0,0,0,0.0,0.001503
191097,QCE2_SW_SGPRF,2027-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NPD,1,0.0,0.0,0.0,0.0,0,0,0.0,0.001503


In [26]:
all = all.sort_values(['channel','key', 'month_date'])

# base LY


# LY lags
all['ly_lag1_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(13)
)

all['ly_lag2_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(14)
)

# LY leads
all['ly_lead1_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(11)
)

all['ly_lead2_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(10)
)


In [27]:
all[all.select_dtypes(include='number').columns] = all.select_dtypes(include='number').fillna(0)
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,price_off_co,seasonal_month_flag,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178623,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.000000,0.003014,0.000000
178624,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.000000,0.000000,0.002078
178625,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.0,0.0,0,-1,0.0,0.005418,0.003014,0.000000,0.002078,0.000000
178626,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.003014,0.000000,0.000000


In [ ]:
# all.to_csv('break.csv')

In [28]:
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,price_off_co,seasonal_month_flag,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,0.0,0.0,0,0,0.0,0.002300,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178623,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.000000,0.003014,0.000000
178624,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.000000,0.000000,0.002078
178625,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.0,0.0,0,-1,0.0,0.005418,0.003014,0.000000,0.002078,0.000000
178626,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.0,0.0,0,-1,0.0,0.005418,0.000000,0.003014,0.000000,0.000000


In [29]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

all['month_date'] = pd.to_datetime(all['month_date'])
all['month'] = all['month_date'].dt.month
all = all.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
all['is_seasonal_month'].fillna(0, inplace=True)
all['final_seasonal_month'] = all['is_seasonal_month']

# psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
# psku_seas.columns = psku_seas.columns.str.lower()
# psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

# all = all.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
# all['is_seasonal_month_psku'].fillna(0, inplace=True)
# all['final_seasonal_month'] = np.where(
#     (all['is_seasonal_month'] == 1) | (all['is_seasonal_month_psku'] == 1), 1, 0
# )



In [30]:
all['run_month'] = pd.to_datetime(all['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = all.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month_value',0)
    })
).reset_index()
adj_df


adj_ly_df = all.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"sec_vol_actuals_rum_month_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,AURG_ADV-AHO-R,2026-03-31,19.990000,28.429167,0.000900,0.001280,389.191667,196.704167,0.017517,0.008853
1,AURG_BIO OILS,2026-03-31,1.151667,1.510000,0.000446,0.000584,1.561667,1.931667,0.000604,0.000748
2,AURG_BRD_BDOIL,2026-03-31,0.010000,0.015000,0.000006,0.000009,0.000000,0.015000,0.000000,0.000009
3,AURG_BRD_BDSPR,2026-03-31,0.000000,0.000000,0.000000,0.000000,0.000000,5.075000,0.000000,0.000513
4,AURG_BRD_DOGAS,2026-03-31,0.000000,0.075000,0.000000,0.000006,0.050000,5.225000,0.000004,0.000435
...,...,...,...,...,...,...,...,...,...,...
4232,WUP_SF_SOYACN,2026-03-31,4.541667,6.196500,0.058965,0.080450,5.053667,6.576333,0.065612,0.085381
4233,WUP_SW HRGEL,2026-03-31,550.778667,383.216000,0.025584,0.017801,919.930000,619.839167,0.042731,0.028792
4234,WUP_SW HSPRY,2026-03-31,0.000000,0.000000,0.000000,0.000000,0.075000,0.150000,0.000005,0.000009
4235,WUP_SW STLDEO,2026-03-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [31]:
#adj_df.to_csv('seasonal_p3m_mt.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
all = all.merge(
    adj_df,
    on=['key'],
    how="left"
)
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,BCE1_ADV-AHO-R,2023-01-31,15.00,12.05,22.887669,15.2760,0.000675,0.000542,0.001030,0.000688,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
1,BCE1_ADV-AHO-R,2023-02-28,15.00,12.05,10.900028,15.1320,0.000675,0.000542,0.000491,0.000681,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
2,BCE1_ADV-AHO-R,2023-03-31,15.00,12.05,7.686881,12.1800,0.000675,0.000542,0.000346,0.000548,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
3,BCE1_ADV-AHO-R,2023-04-30,15.00,12.05,2.505327,10.0308,0.000675,0.000542,0.000113,0.000451,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
4,BCE1_ADV-AHO-R,2023-05-31,9.00,12.05,13.166723,16.1580,0.000405,0.000542,0.000593,0.000727,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191094,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191095,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191096,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191097,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113


In [ ]:
# all[(all['month']==4) & (all['final_seasonal_month'] == 1)]['brand_code'].unique()

array(['REV.LQDST', 'REV.ST.', 'SW STLDEO'], dtype=object)

In [32]:
all[(all['final_seasonal_month'] == 1)]['brand_code'].unique()

array(['PA-BDYLOT', 'PADV-HOT', 'REV.LQDST', 'REV.ST.', 'SAF_HONEY',
       'SW STLDEO', 'NHR_SSBHO', 'NHR_SSAHO'], dtype=object)

In [ ]:
# all[all['brand_code']=='PADV-HOT'].groupby(['key'])['sec_vol_actuals_rum_month_value'].sum().reset_index().sort_values('sec_vol_actuals_rum_month_value', ascending=False)

,key,sec_vol_actuals_rum_month_value
30,DELM_D115_718449,0.877929
29,DELM_D115_718447,0.669892
91,PJB_D117_718449,0.500129
90,PJB_D117_718447,0.421274
47,HAR_D117_718449,0.354434
...,...,...
1,AURG_D3A4_718450,0.000524
116,TPT_D572_719083,0.000342
35,EMP_D463_719083,0.000342
14,BLR_D673_719083,0.000171


In [ ]:
# all[all['key']=='DELM_D115_718449'].to_csv('ses_chek.csv')

In [26]:
all[all['month_date']<='2026-01-31']

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
98688,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.042078,0.0223,0.000456,0.000908,0.001028,0.000545,...,0.0,0.0,0,-1,0.0,0.001398,0.0,0.0,0.000000,0.000000
98689,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.025592,0.0223,0.000456,0.000908,0.000625,0.000545,...,0.0,0.0,0,-1,0.0,0.001398,0.0,0.0,0.000000,0.000000
98690,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.043581,0.0180,0.000456,0.000908,0.001065,0.000440,...,0.0,0.0,0,-1,0.0,0.001398,0.0,0.0,0.000000,0.000000
98691,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.044465,0.0126,0.000456,0.000908,0.001087,0.000308,...,0.0,0.0,0,-1,0.0,0.001398,0.0,0.0,0.000000,0.000000
98692,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.043702,0.0180,0.000912,0.000908,0.001068,0.000440,...,0.0,0.0,0,-1,0.0,0.001398,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215346,MTS_D674_719008,2025-09-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.000000,0.0,0.0,0.000000,0.000000
215347,MTS_D674_719008,2025-10-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.000000,0.0,0.0,0.000000,0.000000
215348,MTS_D674_719008,2025-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.000000,0.0,0.0,0.000000,0.000000
215349,MTS_D674_719008,2025-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.000000,0.0,0.0,0.000000,0.001239


### co p3m

In [5]:
query = """select * from TRN_MIL_CO_DATA where month_date >= '2025-01-31'"""
co_df = pd.read_sql(query, con=dev_conn)
co_df

/tmp/ipykernel_27216/1607714774.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  co_df = pd.read_sql(query, con=dev_conn)


,CHANNEL,MONTH_DATE,ASM_AREA_CODE,PARENT_MATERIAL_CODE,CO_DESC,EXTRA_VOL_CO,OTHER_CO,PRICE_OFF_CO,UPDATE_TIMESTAMP
0,GT,2025-01-31,AURG,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27
1,GT,2025-01-31,BIHE,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27
2,GT,2025-01-31,BIHW,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27
3,GT,2025-01-31,BLR,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27
4,GT,2025-01-31,CMB,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27
...,...,...,...,...,...,...,...,...,...
3043,GT,2026-02-28,RWBS,731588,CO: 60 gm Free,False,True,False,2026-02-05
3044,GT,2026-02-28,ORS,731588,CO: 60 gm Free,False,True,False,2026-02-05
3045,GT,2026-03-31,RWBN,731588,CO: 60 gm Free,False,True,False,2026-02-05
3046,GT,2026-03-31,RWBS,731588,CO: 60 gm Free,False,True,False,2026-02-05


In [6]:
co_df.columns = co_df.columns.str.lower()
co_df.columns

Index(['channel', 'month_date', 'asm_area_code', 'parent_material_code',
       'co_desc', 'extra_vol_co', 'other_co', 'price_off_co',
       'update_timestamp'],
      dtype='object')

In [12]:
all = pd.read_csv('break.csv')
all

,Unnamed: 0,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,...,npd,update_timestamp,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
0,217486,AURG_D3A4_718288,2023-01-31,2.584667,2.537333,2.867240,3.056333,0.035581,0.034930,0.039471,...,0.0,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0
1,217487,AURG_D3A4_718288,2023-02-28,2.584667,2.537333,2.029731,2.785000,0.035581,0.034930,0.027942,...,0.0,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0
2,217488,AURG_D3A4_718288,2023-03-31,2.584667,2.537333,2.053426,3.024167,0.035581,0.034930,0.028268,...,0.0,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0
3,217489,AURG_D3A4_718288,2023-04-30,2.584667,2.537333,2.448877,3.020000,0.035581,0.034930,0.033712,...,0.0,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0
4,217490,AURG_D3A4_718288,2023-05-31,2.398333,2.537333,2.211145,2.944000,0.033016,0.034930,0.030439,...,0.0,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360128,264948,WUP_D113_809238,2026-06-30,0.000667,0.000333,0.045811,0.000300,0.000014,0.000007,0.000986,...,0.0,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0
360129,264949,WUP_D113_809238,2026-07-31,0.000667,0.000333,0.000000,0.000700,0.000014,0.000007,0.000000,...,0.0,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0
360130,264950,WUP_D113_809238,2026-08-31,0.000667,0.000333,0.000000,0.003900,0.000014,0.000007,0.000000,...,0.0,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0
360131,264951,WUP_D113_809238,2026-09-30,0.000667,0.000333,0.000000,0.000500,0.000014,0.000007,0.000000,...,0.0,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0


In [14]:
all['co_flag'] = ((all['extra_vol_co'] == 1) | (all['other_co'] == 1) | (all['price_off_co'] == 1)).astype(int)
all

,Unnamed: 0,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,...,update_timestamp,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,co_flag
0,217486,AURG_D3A4_718288,2023-01-31,2.584667,2.537333,2.867240,3.056333,0.035581,0.034930,0.039471,...,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0,0
1,217487,AURG_D3A4_718288,2023-02-28,2.584667,2.537333,2.029731,2.785000,0.035581,0.034930,0.027942,...,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0,0
2,217488,AURG_D3A4_718288,2023-03-31,2.584667,2.537333,2.053426,3.024167,0.035581,0.034930,0.028268,...,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0,0
3,217489,AURG_D3A4_718288,2023-04-30,2.584667,2.537333,2.448877,3.020000,0.035581,0.034930,0.033712,...,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0,0
4,217490,AURG_D3A4_718288,2023-05-31,2.398333,2.537333,2.211145,2.944000,0.033016,0.034930,0.030439,...,0.0,0,1,0.036461,0.080563,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360128,264948,WUP_D113_809238,2026-06-30,0.000667,0.000333,0.045811,0.000300,0.000014,0.000007,0.000986,...,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0,0
360129,264949,WUP_D113_809238,2026-07-31,0.000667,0.000333,0.000000,0.000700,0.000014,0.000007,0.000000,...,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0,0
360130,264950,WUP_D113_809238,2026-08-31,0.000667,0.000333,0.000000,0.003900,0.000014,0.000007,0.000000,...,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0,0
360131,264951,WUP_D113_809238,2026-09-30,0.000667,0.000333,0.000000,0.000500,0.000014,0.000007,0.000000,...,0.0,0,0,0.000000,0.000028,0.0,0.0,0.0,0.0,0


In [15]:
all[all['co_flag'] == 1]

,Unnamed: 0,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,...,update_timestamp,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,co_flag
22336,356270,BIHE_D232_808712,2025-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.000801,0.0,0.0,0.0,0.0,1
22337,356271,BIHE_D232_808712,2025-10-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.000801,0.0,0.0,0.0,0.0,1
22338,356272,BIHE_D232_808712,2025-11-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.000801,0.0,0.0,0.0,0.0,1
22339,356273,BIHE_D232_808712,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.000801,0.0,0.0,0.0,0.0,1
36470,356336,BIHW_D233_808731,2025-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.001377,0.0,0.0,0.0,0.0,1
36471,356337,BIHW_D233_808731,2025-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.001377,0.0,0.0,0.0,0.0,1
102133,347186,EMP_D464_724031,2025-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.001854,0.0,0.0,0.0,0.0,1
102136,347189,EMP_D464_724031,2025-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000000,0.001854,0.0,0.0,0.0,0.0,1
102647,356367,EMP_D464_808712,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0,0.000047,0.000187,0.0,0.0,0.0,0.0,1


In [10]:
co_df = co_df[co_df['co_flag'] == 1]
co_df = co_df[co_df['channel'] == 'GT']
co_df

,channel,month_date,asm_area_code,parent_material_code,co_desc,extra_vol_co,other_co,price_off_co,update_timestamp,co_flag
0,GT,2025-01-31,AURG,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27,1
1,GT,2025-01-31,BIHE,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27,1
2,GT,2025-01-31,BIHW,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27,1
3,GT,2025-01-31,BLR,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27,1
4,GT,2025-01-31,CMB,718464,CO: Rs. 60 cashback,False,True,False,2024-12-27,1
...,...,...,...,...,...,...,...,...,...,...
3043,GT,2026-02-28,RWBS,731588,CO: 60 gm Free,False,True,False,2026-02-05,1
3044,GT,2026-02-28,ORS,731588,CO: 60 gm Free,False,True,False,2026-02-05,1
3045,GT,2026-03-31,RWBN,731588,CO: 60 gm Free,False,True,False,2026-02-05,1
3046,GT,2026-03-31,RWBS,731588,CO: 60 gm Free,False,True,False,2026-02-05,1


In [16]:
co_df['asm_area_code'].unique()

array(['AURG', 'BIHE', 'BIHW', 'BLR', 'CMB', 'CNI', 'CUP', 'DELM', 'EMP',
       'EUP', 'GUJN', 'GUJS', 'HAR', 'HUB', 'HYD1', 'HYD2', 'JHR', 'KOL',
       'KRL', 'MUM1', 'MUM2', 'NAG', 'NERE', 'ORS', 'PJB', 'PUNN', 'PUNS',
       'RAJ1', 'RAJ2', 'RWBN', 'RWBS', 'TPT', 'VIJ', 'WMP', 'WUP', 'NERW',
       'KARC', 'KARN', 'NUP', 'CTGH', 'ALL'], dtype=object)

In [24]:
realignment_df = pd.read_sql(
    'select * from trn_mil_asm_psku_realignment',
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()
def demand_driver_realign_pskus(data):
        """
        Realign the old pskus to new pskus and return updated data.

        Args:
            data: pandas dataframe
            - master dataframe having all the pskus
        
        Return:
            data: pandas dataframe
            - dataframe 
        """
        realignment_data = realignment_df.copy()
        realignment_data.columns = realignment_data.columns.str.lower()
        realignment_data = realignment_data[
            (realignment_data["channel"] == 'GT')
            | (realignment_data["channel"] == 'GT' + " B2C")
            | (realignment_data["channel"] == "ALL")
        ]

        data["parent_material_code"] = data["parent_material_code"].astype(int)

        for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
            old_psku, old_asm = grp
            new_psku = grp_data["psku new"].values[0]
            if old_asm != "ALL":
                condition = (data["parent_material_code"] == old_psku) & (
                    data["asm_area_code"] == old_asm
                )
            else:
                condition = data["parent_material_code"] == old_psku

            data.loc[condition, "parent_material_code"] = new_psku

        return data

/tmp/ipykernel_27216/4072188734.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  realignment_df = pd.read_sql(


In [25]:
co_data = co_df.rename( columns= {'regionhierarchy':'asm_area_code'})
co_data = co_data.replace( { True:1 , False:0})
co_data = co_data[[ "month_date", 'asm_area_code', "parent_material_code", 'extra_vol_co', 'price_off_co', 'other_co']]
co_data = demand_driver_realign_pskus(co_data)
co_data = co_data.groupby(["month_date", 'asm_area_code', "parent_material_code"], as_index=False).max()
co_data["month_date"] = pd.to_datetime(co_data["month_date"])


### (old_asm, new_asm)
# asm_change_info = [('HUB', 'KARC'), ('HUB', 'KARN'), ('WUP', 'NUP'), ('TPT', 'KUR')]
asm_change_info = {
    'KARC': {'old_asm': 'HUB', 'data_available_from': '2025-01-31'},
    'KARN': {'old_asm': 'HUB', 'data_available_from': '2025-01-31'},
    'NUP': {'old_asm': 'WUP', 'data_available_from': '2025-01-31'},
    'KUR': {'old_asm': 'TPT', 'data_available_from': None}
}


for new_asm in asm_change_info.keys():
    old_asm = asm_change_info[new_asm]['old_asm']
    data_available_from = asm_change_info[new_asm]['data_available_from']

    if data_available_from is not None:
        old_asm_df = co_data[
            (co_data['asm_area_code'] == old_asm) &
            (co_data['month_date'] < pd.to_datetime(data_available_from))
        ]
    else:
        old_asm_df = co_data[
            (co_data['asm_area_code'] == old_asm)
        ]

    new_asm_df = old_asm_df.copy()
    new_asm_df['asm_area_code'] = new_asm
    co_data = pd.concat([co_data, new_asm_df])
    
co_data = co_data.drop_duplicates()
assert co_data.duplicated(
    subset=['asm_area_code', 'parent_material_code', 'month_date']
).sum() == 0
###

# co_data = co_data.reset_index(drop=True)
co_data = co_data.fillna(0)
all["month_date"] = pd.to_datetime(all["month_date"])

all = pd.merge(
    all,
    co_data,
    on=["month_date", 'asm_area_code',"parent_material_code"],
    how="left",
)

In [32]:
all = all.fillna(0)
all['co_flag'] = ((all['extra_vol_co_y'] == 1) | (all['other_co_y'] == 1) | (all['price_off_co_y'] == 1)).astype(int)
all[all['co_flag'] == 1]

,Unnamed: 0,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,...,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,co_flag,extra_vol_co_y,price_off_co_y,other_co_y
267,166969,AURG_D3A4_718309,2026-02-28,14.660333,15.113333,19.225239,19.3986,0.454127,0.468159,0.595532,...,0.247591,1.083158,0.186851,0.542338,0.698553,0.816543,1,0.0,0.0,1.0
451,17209,AURG_D3A4_718317,2026-02-28,117.966667,253.450000,129.926016,125.2168,0.004577,0.009833,0.005041,...,0.000000,0.025476,0.002766,0.005191,0.003744,0.004667,1,0.0,0.0,1.0
576,229656,AURG_D3A4_718322,2025-01-31,0.425000,0.427500,0.378377,0.4111,0.007218,0.007261,0.006427,...,0.003749,0.011414,0.007303,0.003567,0.002208,0.003482,1,0.0,0.0,1.0
1321,136904,AURG_D3A4_718432,2025-10-31,5515.068000,3409.959000,9666.637277,6077.4552,0.164817,0.101906,0.288885,...,0.000000,0.461434,0.197752,0.009188,0.021205,0.007972,1,0.0,0.0,1.0
1323,136906,AURG_D3A4_718432,2025-12-31,8487.882000,5083.686000,1497.056957,1053.3270,0.253659,0.151925,0.044739,...,0.000000,0.461434,0.021205,0.230682,0.008182,0.012574,1,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360010,264830,WUP_D113_808712,2025-10-31,0.065333,0.032833,0.076152,0.0860,0.001407,0.000707,0.001640,...,0.000000,0.006190,0.000000,0.000000,0.000000,0.000000,1,0.0,0.0,1.0
360011,264831,WUP_D113_808712,2025-11-30,0.073667,0.037833,0.079193,0.0769,0.001586,0.000815,0.001705,...,0.000000,0.006190,0.000000,0.000000,0.000000,0.000000,1,0.0,0.0,1.0
360012,264832,WUP_D113_808712,2025-12-31,0.087667,0.044833,0.144759,0.1552,0.001888,0.000965,0.003117,...,0.000000,0.006190,0.000000,0.000000,0.000000,0.000538,1,0.0,0.0,1.0
360072,264892,WUP_D113_808731,2025-03-31,0.006000,0.003000,0.000788,0.0012,0.000129,0.000065,0.000017,...,0.000000,0.000118,0.000000,0.000000,0.000000,0.000000,1,0.0,0.0,1.0


In [33]:
all['run_month'] = pd.to_datetime(all['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["co_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = all.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_co": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',0),
        "P6M_non_co": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month',0),
        "P3M_non_co_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value',0),
        "P6M_non_co_value": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month_value',0)
    })
).reset_index()
adj_df


# adj_ly_df = all.groupby(
#     ['key', "run_month"]
# ).apply(
#     lambda x: pd.Series({
#         "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',year_shift=1),
#         "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month', year_shift=1),
#         "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value', year_shift=1),
#         "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"sec_vol_actuals_rum_month_value", year_shift=1)
#     })
# ).reset_index()

# adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_co,P6M_non_co,P3M_non_co_value,P6M_non_co_value
0,AURG_D3A4_718288,2026-03-31,4.421667,4.235000,0.060870,0.058300
1,AURG_D3A4_718297,2026-03-31,22.612667,19.907667,0.700463,0.616672
2,AURG_D3A4_718299,2026-03-31,31.418000,27.383500,0.973222,0.848247
3,AURG_D3A4_718303,2026-03-31,0.119000,0.119833,0.002908,0.002928
4,AURG_D3A4_718308,2026-03-31,0.036667,0.035667,0.001136,0.001105
...,...,...,...,...,...,...
8979,WUP_D113_808562,2026-03-31,0.164667,0.161333,0.002138,0.002095
8980,WUP_D113_808712,2026-03-31,0.079333,0.040667,0.001708,0.000876
8981,WUP_D113_808713,2026-03-31,0.000000,0.000000,0.000000,0.000000
8982,WUP_D113_808731,2026-03-31,0.000000,0.000000,0.000000,0.000000


In [34]:
adj_df

,key,run_month,P3M_non_co,P6M_non_co,P3M_non_co_value,P6M_non_co_value
0,AURG_D3A4_718288,2026-03-31,4.421667,4.235000,0.060870,0.058300
1,AURG_D3A4_718297,2026-03-31,22.612667,19.907667,0.700463,0.616672
2,AURG_D3A4_718299,2026-03-31,31.418000,27.383500,0.973222,0.848247
3,AURG_D3A4_718303,2026-03-31,0.119000,0.119833,0.002908,0.002928
4,AURG_D3A4_718308,2026-03-31,0.036667,0.035667,0.001136,0.001105
...,...,...,...,...,...,...
8979,WUP_D113_808562,2026-03-31,0.164667,0.161333,0.002138,0.002095
8980,WUP_D113_808712,2026-03-31,0.079333,0.040667,0.001708,0.000876
8981,WUP_D113_808713,2026-03-31,0.000000,0.000000,0.000000,0.000000
8982,WUP_D113_808731,2026-03-31,0.000000,0.000000,0.000000,0.000000


In [35]:
adj_df.to_csv('non_co_p3m.csv')

### co p3m end

In [33]:
all[all['month_date']<='2026-02-28'].to_csv('/data/aman_singh/acuuracy_check/all_combination_brand_asm_trend.csv')

In [29]:
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.043834,0.0180,0.000456,0.000908,0.001071,0.000440,...,0,2026-03-31,0.013667,0.016167,0.000334,0.000395,0.004667,0.021,0.000114,0.000513
1,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.029034,0.0152,0.000456,0.000908,0.000710,0.000371,...,0,2026-03-31,0.013667,0.016167,0.000334,0.000395,0.004667,0.021,0.000114,0.000513
2,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.045455,0.0207,0.000456,0.000908,0.001111,0.000506,...,1,2026-03-31,0.013667,0.016167,0.000334,0.000395,0.004667,0.021,0.000114,0.000513
3,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.045607,0.0264,0.000456,0.000908,0.001115,0.000645,...,1,2026-03-31,0.013667,0.016167,0.000334,0.000395,0.004667,0.021,0.000114,0.000513
4,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.044836,0.0181,0.000912,0.000908,0.001096,0.000442,...,1,2026-03-31,0.013667,0.016167,0.000334,0.000395,0.004667,0.021,0.000114,0.000513
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228058,MCW2_D463_811220,2026-06-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,2026-03-31,0.046000,0.046000,0.000000,0.000000,NaN,NaN,NaN,NaN
228059,MCW2_D463_811220,2026-07-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,2026-03-31,0.046000,0.046000,0.000000,0.000000,NaN,NaN,NaN,NaN
228060,MCW2_D463_811220,2026-08-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,2026-03-31,0.046000,0.046000,0.000000,0.000000,NaN,NaN,NaN,NaN
228061,MCW2_D463_811220,2026-09-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,2026-03-31,0.046000,0.046000,0.000000,0.000000,NaN,NaN,NaN,NaN


In [34]:
all['M month'].unique()
all[all['M month']!='0']

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
38,BCE1_ADV-AHO-R,2026-03-31,26.40,22.20,11.217182,18.1440,0.001188,0.000999,0.000505,0.000817,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
39,BCE1_ADV-AHO-R,2026-04-30,26.40,22.20,6.169579,15.9840,0.001188,0.000999,0.000278,0.000719,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
40,BCE1_ADV-AHO-R,2026-05-31,26.40,22.20,16.809993,16.6740,0.001188,0.000999,0.000757,0.000750,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
41,BCE1_ADV-AHO-R,2026-06-30,26.40,22.20,13.057868,11.1660,0.001188,0.000999,0.000588,0.000503,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
42,BCE1_ADV-AHO-R,2026-07-31,26.40,22.20,19.205087,11.8848,0.001188,0.000999,0.000864,0.000535,...,0.0,2026-03-31,18.00,13.20,0.000810,0.000594,1.80,21.60,0.000081,0.000972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191094,QCW2_SW_SGPRF,2026-11-30,11.76,5.88,0.000000,2.0800,0.001697,0.000849,0.000000,0.000300,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191095,QCW2_SW_SGPRF,2026-12-31,11.76,5.88,0.000000,10.5840,0.001697,0.000849,0.000000,0.001528,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191096,QCW2_SW_SGPRF,2027-01-31,11.76,5.88,0.000000,3.2960,0.001697,0.000849,0.000000,0.000476,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113
191097,QCW2_SW_SGPRF,2027-02-28,11.76,5.88,0.000000,6.6320,0.001697,0.000849,0.000000,0.000957,...,0.0,2026-03-31,11.76,5.88,0.001697,0.000849,49.28,49.28,0.007113,0.007113


### Current month different logic

In [4]:
all = pd.read_excel("/data/aman_singh/acuuracy_check/all_combination_GT_live_mar.xlsx", sheet_name='Base')
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,AURG_D3A4_718288,2026-03-31,4.421667,4.235000,3.897958,4.029000,0.060870,0.058300,0.053660,0.055464,...,0,2026-03-31,4.421667,4.235000,0.060870,0.058300,3.698333,3.804167,0.050912,0.052369
1,AURG_D3A4_718288,2026-04-30,4.421667,4.235000,4.296949,4.143000,0.060870,0.058300,0.059153,0.057034,...,0,2026-03-31,4.421667,4.235000,0.060870,0.058300,3.698333,3.804167,0.050912,0.052369
2,AURG_D3A4_718288,2026-05-31,4.421667,4.235000,4.069224,4.144833,0.060870,0.058300,0.056018,0.057059,...,0,2026-03-31,4.421667,4.235000,0.060870,0.058300,3.698333,3.804167,0.050912,0.052369
3,AURG_D3A4_718288,2026-06-30,4.421667,4.235000,4.343134,4.230500,0.060870,0.058300,0.059789,0.058238,...,0,2026-03-31,4.421667,4.235000,0.060870,0.058300,3.698333,3.804167,0.050912,0.052369
4,AURG_D3A4_718288,2026-07-31,4.421667,4.235000,4.072970,4.303500,0.060870,0.058300,0.056070,0.059243,...,0,2026-03-31,4.421667,4.235000,0.060870,0.058300,3.698333,3.804167,0.050912,0.052369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71867,WUP_D113_809238,2026-06-30,0.000667,0.000333,0.045811,0.000300,0.000014,0.000007,0.000986,0.000006,...,0,2026-03-31,0.000333,0.000167,0.000007,0.000004,0.000000,0.000000,0.000000,0.000000
71868,WUP_D113_809238,2026-07-31,0.000667,0.000333,0.000000,0.000700,0.000014,0.000007,0.000000,0.000015,...,0,2026-03-31,0.000333,0.000167,0.000007,0.000004,0.000000,0.000000,0.000000,0.000000
71869,WUP_D113_809238,2026-08-31,0.000667,0.000333,0.000000,0.003900,0.000014,0.000007,0.000000,0.000084,...,0,2026-03-31,0.000333,0.000167,0.000007,0.000004,0.000000,0.000000,0.000000,0.000000
71870,WUP_D113_809238,2026-09-30,0.000667,0.000333,0.000000,0.000500,0.000014,0.000007,0.000000,0.000011,...,0,2026-03-31,0.000333,0.000167,0.000007,0.000004,0.000000,0.000000,0.000000,0.000000


In [ ]:
def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
    """
    Detect if April's sec_vol_actuals_rum_month_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'sec_vol_actuals_rum_month_value', 'run_month'
    - brand_code: filter by this brand code
    - threshold: 25% difference threshold
    - months_window: number of months before and after April to compare
    """
    
    # Filter for brand and sort by month_date
    df_brand = df[df['key'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    # Extract year and month
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    # Get unique years (excluding current year if incomplete)
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
    anomalies = []
    
    # Check each past year's April
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        # Get April data (month == 4)
        april_data = df_year[df_year['month'] == 4]
        if april_data.empty:
            continue
        
        april_value = april_data['sec_vol_actuals_rum_month_value'].iloc[0]
        april_month = 4
        
        # Dynamically calculate past and next months
        past_months = [(april_month - i - 1) % 12 + 1 for i in range(months_window)]
        next_months = [(april_month + i) % 12 + 1 for i in range(1, months_window + 1)]
        
        # Get past and next months values
        past_3m = df_year[df_year['month'].isin(past_months)]['sec_vol_actuals_rum_month_value']
        next_3m = df_year[df_year['month'].isin(next_months)]['sec_vol_actuals_rum_month_value']
        
        # Combine all comparison months
        comparison_values = pd.concat([past_3m, next_3m])
        
        if comparison_values.empty:
            continue
        
        # Calculate mean of comparison months
        #mean_value = comparison_values.mean()
        
        # Calculate percentage difference
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = abs(april_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        # April is anomalous if it's >25% different from ALL comparison months
        is_anomaly = all(pct_diff > threshold for pct_diff in pct_diffs) if pct_diffs else False
        
        anomalies.append({
            'brand_code': brand_code,
            'year': year,
            'april_value': april_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
        })
    
    # Check if pattern repeats in both years
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Usage: Apply to each brand code
brands = all['key'].unique()
results = []

for brand in brands:
    df_result, repeats = detect_april_anomaly(all, brand)
    if df_result is not None and not df_result.empty:
        df_result['pattern_repeats'] = repeats
        results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [35]:
with pd.ExcelWriter('all_combination_brand_asm_live_mar.xlsx', engine='xlsxwriter') as writer:
    all[all['M month'] != '0'].to_excel(
        writer,
        sheet_name='Base',
        index=False
    )
    
    trend_df.to_excel(
        writer,
        sheet_name='threshold',
        index=False
    )
    
    


In [29]:
all[all['M month'].notna()].to_csv('/data/aman_singh/acuuracy_check/all_combination_MT_live_feb.csv')

In [2]:
import pandas as pd
all = pd.read_csv('/data/aman_singh/acuuracy_check/all_combination_all_channels_pred3.csv')
all

,Unnamed: 0,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,...,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
0,97238,BCE1_D231_718297,2023-01-31,0.223000,0.121,0.097515,0.022833,0.006908,0.003748,0.003021,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
1,97239,BCE1_D231_718297,2023-02-28,0.223000,0.121,0.119242,0.014767,0.006908,0.003748,0.003694,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
2,97240,BCE1_D231_718297,2023-03-31,0.223000,0.121,0.165967,0.033167,0.006908,0.003748,0.005141,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
3,97241,BCE1_D231_718297,2023-04-30,0.223000,0.121,0.121937,0.036867,0.006908,0.003748,0.003777,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
4,97242,BCE1_D231_718297,2023-05-31,0.216667,0.121,0.083432,0.028267,0.006712,0.003748,0.002584,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547679,532698,QCW2_D463_810674,2026-02-28,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
547680,532699,QCW2_D463_810674,2026-03-31,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
547681,532700,QCW2_D463_810674,2026-04-30,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
547682,532701,QCW2_D463_810674,2026-05-31,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0


In [25]:
all

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
97238,BCE1_D231_718297,2023-01-31,0.223000,0.121,0.097515,0.022833,0.006908,0.003748,0.003021,0.000707,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
97239,BCE1_D231_718297,2023-02-28,0.223000,0.121,0.119242,0.014767,0.006908,0.003748,0.003694,0.000457,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
97240,BCE1_D231_718297,2023-03-31,0.223000,0.121,0.165967,0.033167,0.006908,0.003748,0.005141,0.001027,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
97241,BCE1_D231_718297,2023-04-30,0.223000,0.121,0.121937,0.036867,0.006908,0.003748,0.003777,0.001142,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
97242,BCE1_D231_718297,2023-05-31,0.216667,0.121,0.083432,0.028267,0.006712,0.003748,0.002584,0.000876,...,0.0,0.0,0,0,0.0,0.001429,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532698,QCW2_D463_810674,2026-02-28,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
532699,QCW2_D463_810674,2026-03-31,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
532700,QCW2_D463_810674,2026-04-30,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0
532701,QCW2_D463_810674,2026-05-31,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.006011,0.0,0.0,0.0,0.0


In [23]:
all = all.sort_values(['channel','key', 'month_date'])

# base LY


# LY lags
all['ly_lag1_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(13)
)

all['ly_lag2_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(14)
)

# LY leads
all['ly_lead1_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(11)
)

all['ly_lead2_value'] = (
    all
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(10)
)


In [24]:
all = all.fillna(0)

In [54]:
all[['key','month_date','ly_lag1_value','ly_lag2_value','ly_lead1_value','ly_lead2_value']].to_csv('lead_chk.csv')

In [50]:
all[all['key']=='AURG_D3A4_718288'][['month_date','sec_vol_actuals_rum_month_value','ly_lag1_value','ly_lead1_value']]

,month_date,sec_vol_actuals_rum_month_value,ly_lag1_value,ly_lead1_value
205621,2023-01-31,0.043763,NaN,NaN
205622,2023-02-28,0.021888,NaN,NaN
205623,2023-03-31,0.041092,NaN,NaN
205624,2023-04-30,0.036068,NaN,NaN
205625,2023-05-31,0.032351,NaN,NaN
205626,2023-06-30,0.034416,NaN,NaN
205627,2023-07-31,0.034760,NaN,NaN
205628,2023-08-31,0.036756,NaN,NaN
205629,2023-09-30,0.039096,NaN,NaN
205630,2023-10-31,0.038546,NaN,NaN
